<a href="https://colab.research.google.com/github/Chimera418/dl-endsem-logical-ad/blob/main/notebooks/05_colab_final_integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Integration & Progressive Ablation (`E20`, `E21`, `E22`)
**Project:** 22AI304 / 22AIE304 End-Semester Capstone — Team 11  
**Progressive Integration of Proven Winners:**
- **Phase B Winner:** `PDN-S` (`89.64%` Mean / `84.07%` Pooled)
- **Phase C Winner:** `E15_CrissCross` (`90.74%` Mean / `85.33%` Pooled)
- **Phase D Winner:** `E16_FPN` (`90.33%` Mean / `87.84%` All-Time Record Pooled)
- **Phase E Co-Winners:** `E18_Transformer` (`90.26%` / `85.40%` Pooled) & `E19_Mamba` (`90.25%` / `87.33%` Pooled)

**Experiments in this notebook:**
1. **`E20_FPN_CrissCross`**: `PDN-S + E16_FPN + E15_CrissCross`
2. **`E21_FPN_CrissCross_Transformer`**: `PDN-S + E16_FPN + E15_CrissCross + E18_Transformer`
3. **`E22_FPN_CrissCross_Mamba`**: `PDN-S + E16_FPN + E15_CrissCross + E19_Mamba` *(since E18 `90.26%` & E19 `90.25%` tied within `0.01 pp` and Mamba won Pooled by `+1.93 pp` & 3/5 categories)*

*(After locking in the single final winner here, run `06_colab_individual_winner.ipynb` for the 5-category individual benchmark.)*

In [1]:
#@title 1. Verify Colab GPU, RAM, Disk & Install Dependencies
!pip install -q timm thop pyyaml scikit-learn pandas
import torch, psutil, shutil, sys
print('=== COLAB SESSION AUDIT (FINAL INTEGRATION) ===')
print(f'Python        : {sys.version.split()[0]}')
print(f'PyTorch       : {torch.__version__} | CUDA: {torch.version.cuda}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    native_bf16 = p.major >= 8
    print(f'GPU Model     : {p.name} (CC {p.major}.{p.minor}) | VRAM: {p.total_memory/1024**3:.2f} GiB | Native BF16: {native_bf16}')
else:
    raise RuntimeError('No GPU attached! Runtime -> Change runtime type -> L4 / G4 / A100 GPU.')
ram = psutil.virtual_memory()
disk = shutil.disk_usage('/content')
print(f'System RAM    : {ram.available/1e9:.1f} GB free / {ram.total/1e9:.1f} GB total')
print(f'Local SSD     : {disk.free/1e9:.1f} GB free / {disk.total/1e9:.1f} GB total')

=== COLAB SESSION AUDIT (FINAL INTEGRATION) ===
Python        : 3.13.16
PyTorch       : 2.14.1+cu130 | CUDA: 13.0
GPU Model     : NVIDIA L4 (CC 8.9) | VRAM: 22.03 GiB | Native BF16: True
System RAM    : 54.7 GB free / 56.9 GB total
Local SSD     : 253.0 GB free / 253.1 GB total


In [2]:
#@title 2. Mount Drive, Pull Latest Repo, Unpack Data & Run Section 3 Audit
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_URL = 'https://github.com/Chimera418/dl-endsem-logical-ad.git'
DRIVE_ROOT = '/content/drive/MyDrive/DL_Endsem_Team11'
DRIVE_RUNS = f'{DRIVE_ROOT}/runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)

if not os.path.exists('/content/Endsem/.git'):
    !git clone {REPO_URL} /content/Endsem
else:
    !git -C /content/Endsem pull --ff-only
%cd /content/Endsem

if not os.path.exists('/content/data/mvtec_loco_anomaly_detection/breakfast_box'):
    if os.path.exists('/content/data/breakfast_box'):
        os.environ['ENDSEM_LOCO_ROOT'] = '/content/data'
    else:
        os.makedirs('/content/data/mvtec_loco_anomaly_detection', exist_ok=True)
        !tar -xf "{DRIVE_ROOT}/data/mvtec_loco_anomaly_detection.tar.xz" -C /content/data/mvtec_loco_anomaly_detection/
        os.environ['ENDSEM_LOCO_ROOT'] = '/content/data/mvtec_loco_anomaly_detection'
else:
    os.environ['ENDSEM_LOCO_ROOT'] = '/content/data/mvtec_loco_anomaly_detection'

if not os.path.exists('/content/data/imagenet_penalty'):
    os.makedirs('/content/data/imagenet_penalty', exist_ok=True)
    !unzip -q "{DRIVE_ROOT}/data/imagenet_penalty_10k.zip" -d /content/data/imagenet_penalty

os.environ['ENDSEM_IMAGENET_ROOT'] = '/content/data/imagenet_penalty'
os.environ['ENDSEM_TEACHER_PATH'] = f'{DRIVE_ROOT}/weights/teacher_pdn_s.pt'

!python scripts/verify_env.py

Mounted at /content/drive
Cloning into '/content/Endsem'...
remote: Enumerating objects: 253, done.
remote: Counting objects: 100% (253/253), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 253 (delta 154), reused 156 (delta 71), pack-reused 0 (from 0)
Receiving objects: 100% (253/253), 211.03 KiB | 12.41 MiB/s, done.
Resolving deltas: 100% (154/154), done.
/content/Endsem
SECTION 3 — ENVIRONMENT & DATASET VERIFICATION AUDIT
1. CUDA Available    : True
2. GPU Model         : NVIDIA L4 (CC 8.9, BF16=True)
3. GPU Memory        : 22.03 GiB (22563 MiB)
4. PyTorch / CUDA    : PyTorch 2.14.1+cu130 | CUDA 13.0 | cuDNN 92800
   Python / Packages : Python 3.13.16 | timm=1.0.31 | thop=0.1.1 | mamba_ssm=None
5. System RAM        : 50.16 GB free / 52.96 GB total (5.3% used)
6. Disk Space        : 223.53 GB free / 235.68 GB total (/content/Endsem)
------------------------------------------------------------------------------
7. Dataset & Weights Accessibility:
   MVTec LOCO A

In [ ]:
#@title 3. Run Progressive Integration (E20_FPN_CrissCross, E21_+Transformer, E22_+Mamba)
!python scripts/run_experiment.py --config configs/E20_fpn_crisscross.yaml --mode train --drive-sync-dir "$DRIVE_RUNS"
!python scripts/run_experiment.py --config configs/E21_fpn_crisscross_transformer.yaml --mode train --drive-sync-dir "$DRIVE_RUNS"
!python scripts/run_experiment.py --config configs/E22_fpn_crisscross_mamba.yaml --mode train --drive-sync-dir "$DRIVE_RUNS"

RUN: E20_FPN_CrissCross | mode=train | device=cuda | seed=42
Dataset audit OK: train=1778 | val=305 | test_good=575 | test_logical=561 | structural_excluded=432
Loading pooled train/good and validation/good (img_size=256, cache_in_ram=True)...
  Penalty corpus ready: 10,000 images (batch_size=8)
  Computing streaming teacher channel normalization stats on pooled train/good...
  Starting training: epochs=200, batch_size=8, amp=torch.bfloat16, trainable_params=7,134,850, frozen_params=2,694,144
  Epoch   1/200 | train=10.4935 | val=7.2080 | lr=3.0e-04 | 83.4s <-- BEST
  Epoch   2/200 | train=6.1660 | val=5.6208 | lr=3.0e-04 | 65.9s <-- BEST
  Epoch   3/200 | train=4.8352 | val=4.4293 | lr=3.0e-04 | 65.7s <-- BEST
  Epoch   4/200 | train=4.1391 | val=3.8263 | lr=3.0e-04 | 65.8s <-- BEST
  Epoch   5/200 | train=3.6810 | val=3.4947 | lr=3.0e-04 | 65.7s <-- BEST
  Epoch   6/200 | train=3.4113 | val=3.2219 | lr=3.0e-04 | 66.1s <-- BEST
  Epoch   7/200 | train=3.1675 | val=3.1994 | lr=3.0e-04 

In [ ]:
#@title 4. Aggregate Complete Master Comparison Table (E01–E22)
!python scripts/aggregate_results.py --dirs experiments "$DRIVE_RUNS" --out-dir "$DRIVE_RUNS/final_summary"